In [1]:
library(RTCGA)
library(RTCGA.clinical) 
library(RTCGA.rnaseq)
library(survival)
library(survminer)
library(dplyr)

# Load datasets
data(PAAD.rnaseq)
data(PAAD.clinical)
library(TCGAbiolinks)


Welcome to the RTCGA (version: 1.36.0). Read more about the project under https://rtcga.github.io/RTCGA/

Loading required package: ggplot2

Loading required package: ggpubr


Attaching package: ‘survminer’


The following object is masked from ‘package:survival’:

    myeloma



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:

get_gene_info <-function(genes){

    # Extract gene symbols from colnames (remove the | and everything after)
#     gene_symbols <- sub("\\|.*", "", colnames(PAAD.rnaseq))
    gene_symbols <-  colnames(PAAD.rnaseq)
    # Find columns that match target genes
    cols_to_use <- which(gene_symbols %in% genes)

    # Subset expression matrix for those columns
    expr_df <- PAAD.rnaseq[, cols_to_use]

    # Rename columns to gene symbols (clean names)
    colnames(expr_df) <- gene_symbols[cols_to_use]

    # Add patient IDs as a column (rownames of PAAD.rnaseq)
    expr_df$patient_id <- PAAD.rnaseq$bcr_patient_barcode
    expr_df$patient_id <- substr(expr_df$patient_id, 1, 12)


    # Prepare clinical data
    clinical <- PAAD.clinical %>%
      mutate(patient_id = patient.bcr_patient_barcode) %>%
      filter(!is.na(patient.days_to_death) | !is.na(patient.days_to_last_followup)) %>%
      mutate(
        time = ifelse(is.na(patient.days_to_death),
                      as.numeric(as.character(patient.days_to_last_followup)),
                      as.numeric(as.character(patient.days_to_death))),
        status = ifelse(is.na(patient.days_to_death), 0, 1)
      ) %>%
      select(patient_id, time, status)


    # Merge expression and clinical data
    expr_df$patient_id <- tolower(expr_df$patient_id)
    clinical$patient_id <- tolower(clinical$patient_id)
    merged <- inner_join(expr_df, clinical, by = "patient_id")

    merged   
}


get_worst_group <- function(fit, method='auc') {
    
    if (method=='auc'){
        if (is.null(fit$strata)) {
        stop("The survfit object must be stratified (i.e., fit by a grouping variable).")
        }

        auc_list <- list()
        group_sizes <- fit$strata
        group_names <- names(group_sizes)
        start_idx <- cumsum(c(1, head(group_sizes, -1)))
        end_idx <- cumsum(group_sizes)

        for (i in seq_along(group_sizes)) {
        surv <- fit$surv[start_idx[i]:end_idx[i]]
        time <- fit$time[start_idx[i]:end_idx[i]]
        auc <- sum(diff(c(0, time)) * surv)
        auc_list[[group_names[i]]] <- auc
        }

        worst_group <- names(auc_list)[which.min(unlist(auc_list))]
        worst_group <- unlist(strsplit(worst_group, "="))[2]
    }
    if (method=='median'){
        summary_data <- summary(fit)
        medians <- summary(fit)$table[, "median"]
        worst_group <- names(medians)[which.min(medians)]
        worst_group <- unlist(strsplit(worst_group, "="))[2]
    }
    return(worst_group)
}


get_all_km <- function(allgenes_info, probs){
    allgenes_res <- allgenes_info
    pvals=c()
    for (gene in all_genes){
        if (sum(allgenes_res[[gene]]) != 0){
            print(gene)
            cutoff <- quantile(allgenes_res[[gene]], probs = probs, na.rm = TRUE)
            allgenes_res$group <- ifelse(allgenes_res[[gene]] > cutoff, "High", "Low")
            fit <- survfit(Surv(time, status) ~ group, data = allgenes_res)
            pval <- surv_pvalue(fit)$pval
            worst_group <- get_worst_group(fit, method='auc')
            if (worst_group=='Low'){
                pval=1
            }
        } else{
            pval=1
        }
        pvals<-append(pvals, pval)
    }
    
    res_df <- data.frame('gene'=all_genes, 'pval'=pvals)
    res_df$logpval <- -log10(res_df$pval)
    res_df$symbols <- sub("\\|.*", "", res_df$gene)

    return(res_df)
}



In [3]:

gene_symbols <- colnames(PAAD.rnaseq)
all_genes <- gene_symbols[31:length(gene_symbols)]
all_genes <- unique(all_genes)

# all_genes <-all_genes[1:20]
allgenes_res <- get_gene_info(all_genes)
colnames(allgenes_res) <- sub("\\|.*", "", colnames(allgenes_res))

In [4]:
epi_genes <- unlist(read.table("/home/xuquan/projects/pdac/scripts/tissue_ref/epi.txt"), use.names = FALSE)
mes_genes <- unlist(read.table("/home/xuquan/projects/pdac/scripts/tissue_ref/mes.txt"), use.names = FALSE)
# mes_genes[1] <- 'C10orf116'

lr_genes<-c('COL1A1','COMP','FN1','HSPG2','COL4A1','LAMB1','NID1','SLIT2',
            'THBS1','APP','C3','CCN1','COL11A1','COL1A2','COL3A1','COL5A2','COL8A1','DCN',
            'ECM1','LAMA2','MDK','SPON1','THBS2','TIMP1','TIMP2','ITGA2','ITGA3','ITGB6',
            'ITGB8','SDC1','DAG1','DDR1','ITGA6','PTPRF','TNFRSF21','CD46','SDC4','MET',
            'ITGB4','TSPAN1','CD63','F11R','ADAM9','EFNA1','LAMC2','SERPINA1','TFPI',
            'VEGFA','APP','AZGP1','DAG1','LAMA3','LGALS3BP','MDK','SLPI','JAM3','ITGAV',
            'ITGB1','ITGB5','EPHA3','LRP1','NRP1','CAV1','LAMA4','SDC2','PLSCR4')

### ESTIMATE

In [ ]:
pudf <- read.table("../db/pancreatic_ductal_adenocarcinoma_RNAseqV2.txt", 
                   header=TRUE, row.names=1)
pudf$purity <- cos(0.6049872018 + 0.0001467884 * pudf$ESTIMATE_score)


In [6]:
PAAD.rnaseq$short_id <- substr(PAAD.rnaseq$bcr_patient_barcode, 1, 15)
PAAD.rnaseq$short_id2 <- substr(PAAD.rnaseq$bcr_patient_barcode, 1, 12)
pudf$short_id        <- substr(rownames(pudf), 1, 15)

purity_dict <- unique(pudf[, c("short_id", "purity")])
PAAD.rnaseq <- merge(PAAD.rnaseq, 
                     purity_dict, 
                     by = "short_id", 
                     all.x = TRUE)
PAAD.rnaseq$short_id <- NULL

PAAD.clinical$short_id2 <- toupper(substr(PAAD.clinical$patient.bcr_patient_barcode, 1, 12))
PAAD.rnaseq <- merge(PAAD.rnaseq, 
                     PAAD.clinical, 
                     by = "short_id2", 
                     all.x = TRUE)


In [7]:
pvals=c()
for (gene in epi_genes){
    if (sum(allgenes_res[[gene]]) != 0){
        cutoff <- quantile(allgenes_res[[gene]], probs = 0.5, na.rm = TRUE)
        allgenes_res$group <- ifelse(allgenes_res[[gene]] > cutoff, "High", "Low")
        allgenes_res$group <- factor(allgenes_res$group, levels = c("Low", "High"))
        allgenes_res$purity <- PAAD.rnaseq$purity
        allgenes_res$gender <- PAAD.rnaseq$patient.gender
        allgenes_res$stage <- PAAD.rnaseq$patient.stage_event.pathologic_stage
        allgenes_res$age <- as.numeric(PAAD.rnaseq$patient.age_at_initial_pathologic_diagnosis)
        allgenes_res$other <- PAAD.rnaseq$patient.clinical_cqcf.history_of_prior_malignancy
        
        allgenes_res$stage_clean <- NA_character_
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage i", "stage ia", "stage ib")] <- "Stage I"
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iia", "stage iib")] <- "Stage II"
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iii", "stage iv")] <- "Stage III-IV"

        fit <- coxph(Surv(time, status) ~ group+purity, data = allgenes_res)
        pval <- summary(fit)$coefficients["groupHigh", "Pr(>|z|)"]
    } else{
        pval=NA
    }
    pvals<-append(pvals, pval)
}

res_df <- data.frame('gene'=epi_genes, 'pval'=pvals)
res_df$logpval <- -log10(res_df$pval)
res_df$symbols <- sub("\\|.*", "", res_df$gene)

write.table(res_df, "/home/xuquan/projects/pdac/revision2/epi_km_50_purity.txt", 
            row.names=FALSE, sep="\t", quote = FALSE)


In [8]:
pvals=c()
for (gene in mes_genes){
    if (sum(allgenes_res[[gene]]) != 0){
        cutoff <- quantile(allgenes_res[[gene]], probs = 0.5, na.rm = TRUE)
        allgenes_res$group <- ifelse(allgenes_res[[gene]] > cutoff, "High", "Low")
        allgenes_res$group <- factor(allgenes_res$group, levels = c("Low", "High"))
        allgenes_res$purity <- PAAD.rnaseq$purity
        allgenes_res$gender <- PAAD.rnaseq$patient.gender
        allgenes_res$stage <- PAAD.rnaseq$patient.stage_event.pathologic_stage
        allgenes_res$age <- as.numeric(PAAD.rnaseq$patient.age_at_initial_pathologic_diagnosis)
        allgenes_res$other <- PAAD.rnaseq$patient.clinical_cqcf.history_of_prior_malignancy
        
        allgenes_res$stage_clean <- NA_character_
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage i", "stage ia", "stage ib")] <- "Stage I"
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iia", "stage iib")] <- "Stage II"
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iii", "stage iv")] <- "Stage III-IV"

        fit <- coxph(Surv(time, status) ~ group+purity, data = allgenes_res)
        pval <- summary(fit)$coefficients["groupHigh", "Pr(>|z|)"]
    } else{
        pval=NA
    }
    pvals<-append(pvals, pval)
}

res_df <- data.frame('gene'=mes_genes, 'pval'=pvals)
res_df$logpval <- -log10(res_df$pval)
res_df$symbols <- sub("\\|.*", "", res_df$gene)

write.table(res_df, "/home/xuquan/projects/pdac/revision2/mes_km_50_purity.txt", 
            row.names=FALSE, sep="\t", quote = FALSE)


In [9]:

pvals=c()
for (gene in lr_genes){
    if (sum(allgenes_res[[gene]]) != 0){
        cutoff <- quantile(allgenes_res[[gene]], probs = 0.5, na.rm = TRUE)
        allgenes_res$group <- ifelse(allgenes_res[[gene]] > cutoff, "High", "Low")
        allgenes_res$group <- factor(allgenes_res$group, levels = c("Low", "High"))
        allgenes_res$purity <- PAAD.rnaseq$purity
        allgenes_res$gender <- PAAD.rnaseq$patient.gender
        allgenes_res$stage <- PAAD.rnaseq$patient.stage_event.pathologic_stage
        allgenes_res$age <- as.numeric(PAAD.rnaseq$patient.age_at_initial_pathologic_diagnosis)
        allgenes_res$other <- PAAD.rnaseq$patient.clinical_cqcf.history_of_prior_malignancy

        allgenes_res$stage_clean <- NA_character_
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage i", "stage ia", "stage ib", "stage iia", "stage iib")] <- "Stage I"
        allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iii", "stage iv")] <- "Stage III-IV"

        fit <- coxph(Surv(time, status) ~ group+purity, data = allgenes_res)
        pval <- summary(fit)$coefficients["groupHigh", "Pr(>|z|)"]
    } else{
        pval=NA
    }
    pvals<-append(pvals, pval)
}

res_df <- data.frame('gene'=lr_genes, 'pval'=pvals)
res_df$logpval <- -log10(res_df$pval)
res_df$symbols <- sub("\\|.*", "", res_df$gene)

write.table(res_df, "/home/xuquan/projects/pdac/revision2/lr_genes_km_50_purity.txt", 
            row.names=FALSE, sep="\t", quote = FALSE)


In [10]:
gene='SDC1'
cutoff <- quantile(allgenes_res[[gene]], probs = 0.5, na.rm = TRUE)
allgenes_res$group <- ifelse(allgenes_res[[gene]] > cutoff, "High", "Low")
allgenes_res$group <- factor(allgenes_res$group, levels = c("Low", "High"))
allgenes_res$purity <- PAAD.rnaseq$purity
allgenes_res$gender <- PAAD.rnaseq$patient.gender
allgenes_res$stage <- PAAD.rnaseq$patient.stage_event.pathologic_stage
allgenes_res$age <- as.numeric(PAAD.rnaseq$patient.age_at_initial_pathologic_diagnosis)
allgenes_res$other <- PAAD.rnaseq$patient.clinical_cqcf.history_of_prior_malignancy

allgenes_res$stage_clean <- NA_character_
allgenes_res$stage_clean[allgenes_res$stage %in% c("stage i", "stage ia", "stage ib", "stage iia", "stage iib")] <- "Stage I"
allgenes_res$stage_clean[allgenes_res$stage %in% c("stage iii", "stage iv")] <- "Stage III-IV"

fit <- coxph(Surv(time, status) ~ group+purity, data = allgenes_res)
pval <- summary(fit)$coefficients["groupHigh", "Pr(>|z|)"]
pval

[1] 0.01591313

In [114]:
write.table(t(allgenes_res), "/home/xuquan/projects/pdac/revision2/TCGA_exp.txt", sep='\t')